In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

In [ ]:
df.head().T

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ','_')

categorical_columns = list(df.dtypes[df.dtypes=='object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
# read about coerce and errors
tc = pd.to_numeric(df.totalcharges, errors='coerce') 

In [ ]:
df[tc.isnull()][['customerid', 'totalcharges']]

In [ ]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce') 
df.totalcharges = df.totalcharges.fillna(0)

In [ ]:
df.churn = (df.churn == 'yes').astype(int)


In [ ]:
df.churn

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df_full_train, df_test = train_test_split(df,test_size=0.2, random_state=1)

In [ ]:
len(df_full_train), len(df_test)

In [ ]:
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [ ]:
len(df_train), len(df_test), len(df_val)

In [ ]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [ ]:
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values
len(y_train), len(y_val), len(y_test)

In [ ]:
del df_train['churn']
del df_val['churn']
del df_test['churn']

In [ ]:
df_full_train = df_full_train.reset_index(drop=True)

In [ ]:
df_full_train

In [ ]:
df_full_train.churn.value_counts(normalize=True)

In [ ]:
global_churn_rate = df_full_train.churn.mean().round(2)
global_churn_rate

In [ ]:
numerical = ['tenure','monthlycharges','totalcharges']

In [ ]:
categorical = ['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
               'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod',  'churn']

In [ ]:
df_full_train[categorical].nunique()

In [ ]:
df_full_train.head()

In [ ]:
churn_male = df_full_train[df_full_train.gender == 'male'].churn.mean()

In [ ]:
churn_female = df_full_train[df_full_train.gender == 'female'].churn.mean()
churn_female

In [ ]:
churn_partner = df_full_train[df_full_train.partner == 'yes'].churn.mean()
churn_no_partner = df_full_train[df_full_train.partner=='no'].churn.mean()

In [ ]:
# DIFFERENCE - global - group , if -ve more likely to churn else not.
global_churn_rate - churn_partner
global_churn_rate - churn_no_partner

In [ ]:
# RISK RATIO - Group/Global, >1 more likely to churn else not
churn_partner/global_churn_rate
churn_no_partner/global_churn_rate  

In [ ]:
# Study about agg and groupby
df_group = df_full_train.groupby('gender').churn.agg(['mean','count'])

In [ ]:
# Study this too
df_group['diff'] = df_group['mean']- global_churn_rate
df_group['risk'] = df_group['mean']/ global_churn_rate
df_group

In [ ]:
from IPython.display import display

for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).churn.agg(['mean','count'])
    df_group['diff'] = df_group['mean']- global_churn_rate
    df_group['risk'] = df_group['mean']/ global_churn_rate
    display(df_group)
    print()


In [ ]:
from sklearn.metrics import mutual_info_score

# read about this
mutual_info_score(df_full_train.churn , df_full_train.contract)

In [ ]:
mutual_info_score(df_full_train.churn , df_full_train.contract)

In [ ]:
mutual_info_score(df_full_train.gender, df_full_train.churn)

In [ ]:
mutual_info_score(df_full_train.partner, df_full_train.churn)

In [ ]:
mutual_info_score(df_full_train.seniorcitizen, df_full_train.churn)

In [ ]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.churn)

In [ ]:
mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

In [ ]:
'''
FEATURE CORRELATION- WAY TO MEASURE MUTUAL IMPORTANCE FOR NUMERICAL
-There is a variable r , -1<=r<=1 , if value is -ve variables are 
 inversely proportional, if +ve varaibles are proportional
'''
df_full_train[numerical].corrwith(df_full_train.churn)

In [ ]:
from sklearn.feature_extraction import DictVectorizer

In [ ]:
'''
Without using orient the dict converts data into cols type dictionary
'''
categorical.remove('churn')
train_dicts = df_train[categorical+numerical].to_dict(orient='records')
train_dicts[0]

In [ ]:
dv = DictVectorizer(sparse=False)

In [ ]:
dv.fit(train_dicts)

In [ ]:
dv.transform(train_dicts)

In [ ]:
X_train = dv.fit_transform(train_dicts)

In [ ]:
X_train.shape

In [ ]:
val_dicts = df_val[categorical + numerical].to_dict(orient='records')

In [ ]:
X_val = dv.fit_transform(val_dicts)

In [ ]:
def sigmoid(z):
    return 1/(1+np.exp(-z))

In [ ]:
z = np.linspace(-5,5,51)
sigmoid(z)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(z, sigmoid(z))

In [ ]:
def logistic_regression(xi):
    score = w0

    for j in range(len(w)):
        score = score + xi[j]*w[j]

    result = sigmoid(z)
    return result

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model = LogisticRegression()

In [ ]:
model.fit(X_train,y_train)

In [ ]:
model.coef_[0]

In [ ]:
model.intercept_[0].round(3)

In [ ]:
y_pred = model.predict_proba(X_train)[:,1]

In [ ]:
churn_decision = (y_pred >= 0.5)

In [ ]:
df_val[churn_decision].customerid

In [ ]:
y_pred

In [ ]:
y_pred = model.predict_proba(X_val)[:,1]